# FortyGuard Heatmap Exploration for CropSage

This notebook tests FortyGuard over a small agricultural polygon near Plainview, Hale County, Texas. It produces the local heat evidence that will later be normalized in Task 5 and consumed by the deterministic recommendation engine.

We test four heatmap modes:

1. `tcm`: temperature context across the farm
2. `exceedance`: total hours above a selected crop threshold
3. `persistence`: longest continuous run above that threshold
4. `time_of_measure`: hour when each tile reaches peak heat

> Each request consumes FortyGuard credits when it completes. Run request cells once, one at a time. The notebook never prints the API key.

## What this notebook must answer

- Does FortyGuard return data for a rural Texas farm?
- What temperature fields and units are returned?
- Does the documented multi-day filter work with the hackathon key?
- Are exceedance and persistence usable with crop-specific thresholds?
- How much spatial variation exists across the farm polygon?
- Which normalized values should be passed to CropSage's scorer?

In [2]:
import json
import os
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import folium
import matplotlib.pyplot as plt
import pandas as pd
from branca.colormap import LinearColormap
from dotenv import load_dotenv
from IPython.display import display
from shapely.geometry import shape

PROJECT_ROOT = Path.cwd().resolve()
QUICKSTART_CANDIDATES = [
    PROJECT_ROOT / '..' / 'FortyGuard 26' / 'temperature-api-quickstart',
    PROJECT_ROOT / 'temperature-api-quickstart',
]
QUICKSTART_ROOT = next((p.resolve() for p in QUICKSTART_CANDIDATES if p.exists()), None)
if QUICKSTART_ROOT is None:
    raise FileNotFoundError('Could not locate the FortyGuard temperature-api-quickstart repository.')

load_dotenv(PROJECT_ROOT / '.env')
load_dotenv(QUICKSTART_ROOT / '.env')
sys.path.insert(0, str(QUICKSTART_ROOT))

from fortyguard import FortyGuardClient

client = FortyGuardClient(timeout=60)
print('Imports successful')
print('Quickstart:', QUICKSTART_ROOT)
print('API base URL:', client.base_url)
print('API key loaded:', bool(os.getenv('FORTYGUARD_API_KEY')))

Imports successful
Quickstart: C:\Users\sumai\OneDrive\Documents\ChatGPT\FortyGuard 26\temperature-api-quickstart
API base URL: https://api.fortyguard.com
API key loaded: True


In [3]:
# Small agricultural AOI near Plainview, Texas: approximately 1 km x 1 km.
FARM_NAME = 'Plainview demonstration farm'
FARM_LAT = 34.1800
FARM_LON = -101.7600
FARM_TIMEZONE = 'America/Chicago'
GRANULARITY_M = 100

FARM_AOI = {
    'type': 'FeatureCollection',
    'features': [{
        'type': 'Feature',
        'properties': {'name': FARM_NAME},
        'geometry': {
            'type': 'Polygon',
            'coordinates': [[
                [-101.7655, 34.1755],
                [-101.7545, 34.1755],
                [-101.7545, 34.1845],
                [-101.7655, 34.1845],
                [-101.7655, 34.1755],
            ]],
        },
    }],
}

texas_today = datetime.now(ZoneInfo(FARM_TIMEZONE)).date()
END_DATE = texas_today - timedelta(days=1)
START_DATE = END_DATE - timedelta(days=6)

print('Farm:', FARM_NAME)
print('Coordinate:', (FARM_LAT, FARM_LON))
print('Analysis window:', START_DATE, 'to', END_DATE, '(7 complete Texas local dates)')
print('Granularity:', GRANULARITY_M, 'm')

Farm: Plainview demonstration farm
Coordinate: (34.18, -101.76)
Analysis window: 2026-08-19 to 2026-08-25 (7 complete Texas local dates)
Granularity: 100 m


In [4]:
aoi_map = folium.Map(
    location=[FARM_LAT, FARM_LON],
    zoom_start=14,
    tiles=None,
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
    overlay=False,
).add_to(aoi_map)

folium.GeoJson(
    FARM_AOI,
    name="Farm AOI",
    style_function=lambda _feature: {
        "color": "#1f6f43",
        "weight": 3,
        "fillColor": "#8fd19e",
        "fillOpacity": 0.20,
    },
).add_to(aoi_map)

folium.Marker(
    [FARM_LAT, FARM_LON],
    tooltip=FARM_NAME,
).add_to(aoi_map)

display(aoi_map)

In [5]:
aoi_map.save("plainview_farm_map.html")

In [6]:
catalog_path = PROJECT_ROOT / 'data' / 'crop-catalog' / 'catalog.json'
catalog = json.loads(catalog_path.read_text(encoding='utf-8'))

threshold_rows = []
for crop in catalog['crops']:
    if 'plains' not in crop.get('supported_texas_regions', []):
        continue
    threshold = crop.get('heat_stress_threshold') or {}
    value_c = threshold.get('value_c')
    if value_c is None:
        continue
    threshold_rows.append({
        'crop_id': crop['crop_id'],
        'crop': crop['common_name'],
        'threshold_c': value_c,
        'severity': threshold.get('severity'),
        'scoring_use': threshold.get('scoring_use'),
        'evidence_status': threshold.get('evidence_status'),
    })

crop_thresholds_df = pd.DataFrame(threshold_rows).sort_values(['threshold_c', 'crop'])
print('Catalog crops:', len(catalog['crops']))
print('Plains crops with numeric heat thresholds:', len(crop_thresholds_df))
display(crop_thresholds_df)

Catalog crops: 22
Plains crops with numeric heat thresholds: 10


,crop_id,crop,threshold_c,severity,scoring_use,evidence_status
9,fresh_market_spinach,Fresh-market spinach,23.9,warning,informational_only,direct
5,grain_oats,Grain oats,27.8,warning,soft_penalty,regional_transfer
4,soybean,Soybean,29.4,warning,soft_penalty,regional_transfer
2,hard_red_winter_wheat,Hard red winter wheat,30.0,warning,soft_penalty,regional_transfer
1,corn_grain,Corn grown for grain,35.0,warning,soft_penalty,regional_transfer
7,corn_silage,Corn grown for silage,35.0,warning,soft_penalty,regional_transfer
8,dry_bulb_onion,Dry-bulb onion,35.0,warning,informational_only,direct
6,oilseed_sunflower,Oilseed sunflower,35.0,warning,informational_only,regional_transfer
3,grain_sorghum,Grain sorghum,37.2,warning,soft_penalty,regional_transfer
0,upland_cotton,Upland cotton,37.8,warning,soft_penalty,regional_transfer


In [7]:
# Change this crop ID to explore a different sourced threshold from the 22-crop catalog.
SELECTED_CROP_ID = 'upland_cotton'
selected_row = crop_thresholds_df.loc[crop_thresholds_df['crop_id'] == SELECTED_CROP_ID]
if selected_row.empty:
    raise ValueError(f'{SELECTED_CROP_ID!r} has no numeric heat threshold for this test.')

SELECTED_CROP_NAME = selected_row.iloc[0]['crop']
HEAT_THRESHOLD_C = float(selected_row.iloc[0]['threshold_c'])
print('Selected crop:', SELECTED_CROP_NAME)
print('Heat threshold:', HEAT_THRESHOLD_C, '°C')
print('This is a screening threshold from the catalog, not a universal biological cutoff.')

Selected crop: Upland cotton
Heat threshold: 37.8 °C
This is a screening threshold from the catalog, not a universal biological cutoff.


In [8]:
responses = {}

def run_heatmap(name, analytic_type, threshold=None, direction=None):
    kwargs = {
        'polygon_aoi': FARM_AOI,
        'start_date': START_DATE.isoformat(),
        'end_date': END_DATE.isoformat(),
        'filter_type': 4,
        'granularity': GRANULARITY_M,
        'analytic_type': analytic_type,
        'poll_interval': 3.0,
        'timeout': 600.0,
        'verbose': True,
    }
    if threshold is not None:
        kwargs['threshold'] = float(threshold)
    if direction is not None:
        kwargs['direction'] = direction

    response = client.create_heatmap(**kwargs)
    responses[name] = response
    result = response['result']
    features = result.get('map_data', {}).get('features', [])
    print('Activity ID:', response['activity_id'])
    print('Result keys:', list(result.keys()))
    print('Returned tiles:', len(features))
    return response

def feature_value_key(result):
    features = result.get('map_data', {}).get('features', [])
    if not features:
        raise ValueError('No map features returned.')
    keys = features[0].get('properties', {}).keys()
    for candidate in ('average_temperature', 'temperature', 'value'):
        if candidate in keys:
            return candidate
    raise KeyError(f'No supported value field. Available fields: {list(keys)}')

def tile_dataframe(response, label):
    result = response['result']
    key = feature_value_key(result)
    rows = []
    for feature in result['map_data']['features']:
        properties = feature.get('properties', {})
        rows.append({
            'analysis': label,
            'tile_id': properties.get('tile_id'),
            'value_key': key,
            'value': pd.to_numeric(properties.get(key), errors='coerce'),
        })
    return pd.DataFrame(rows)

def render_heatmap(response, title, unit, colors=('blue', 'yellow', 'red')):
    result = response['result']
    map_data = result['map_data']
    key = feature_value_key(result)
    values = [
        float(f['properties'][key])
        for f in map_data['features']
        if f.get('properties', {}).get(key) is not None
    ]
    lo, hi = min(values), max(values)
    if hi == lo:
        hi = lo + 1e-9
    color_scale = LinearColormap(list(colors), vmin=lo, vmax=hi, caption=f'{title} ({unit})')

    fmap = folium.Map(location=[FARM_LAT, FARM_LON], zoom_start=14, tiles='OpenStreetMap')
    folium.GeoJson(
        map_data,
        style_function=lambda feature: {
            'fillColor': color_scale(float(feature['properties'][key])),
            'color': '#333333',
            'weight': 0.25,
            'fillOpacity': 0.72,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['tile_id', key],
            aliases=['Tile:', f'{title}:'],
        ),
    ).add_to(fmap)
    color_scale.add_to(fmap)
    print(f'{title}: min={min(values):.2f}, mean={sum(values)/len(values):.2f}, max={max(values):.2f} {unit}')
    display(fmap)

## Live requests

Run the next four cells one at a time. Wait for `Done.` before continuing. If `filter_type=4` is rejected by the live API, keep the error output; it resolves an important documentation conflict and we will switch to individual daily requests.

In [9]:
# Request 1/4: seven-day temperature context.
tcm_response = run_heatmap('tcm', analytic_type='tcm')

Submitted -> activity_id=a0b0f003-cbe9-4012-8f7a-9013b282f806
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Activity ID: a0b0f003-cbe9-4012-8f7a-9013b282f806
Result keys: ['map_data', 'stats_data']
Returned tiles: 68


In [10]:
tcm_result = tcm_response['result']
print('Stats data:')
print(json.dumps(tcm_result.get('stats_data', {}), indent=2)[:6000])
print('\nFirst tile properties:')
print(json.dumps(tcm_result['map_data']['features'][0]['properties'], indent=2))

tcm_tiles_df = tile_dataframe(tcm_response, 'tcm')
display(tcm_tiles_df['value'].describe().to_frame('temperature_c'))
render_heatmap(tcm_response, 'Seven-day average temperature', '°C')

Stats data:
{
  "temperature_stats": {
    "minimum": 29.9143,
    "maximum": 29.9146,
    "mean": 29.914445588235292,
    "standard_deviation": 6.563995763717626e-05
  },
  "overall_temperature_distribution": [
    29.9143,
    29.9144,
    29.9144,
    29.9145,
    29.9146
  ],
  "normal_temperature_distribution": {
    "x_axis": [
      29.91424866836238,
      29.91425264654163,
      29.91425662472088,
      29.914260602900132,
      29.914264581079383,
      29.914268559258634,
      29.914272537437885,
      29.914276515617136,
      29.914280493796387,
      29.914284471975638,
      29.91428845015489,
      29.91429242833414,
      29.91429640651339,
      29.914300384692638,
      29.91430436287189,
      29.91430834105114,
      29.91431231923039,
      29.91431629740964,
      29.914320275588892,
      29.914324253768143,
      29.914328231947394,
      29.914332210126645,
      29.914336188305896,
      29.914340166485147,
      29.914344144664398,
      29.91434812284365,

,temperature_c
count,68.000000
mean,29.914446
std,0.000066
min,29.914300
25%,29.914400
50%,29.914400
75%,29.914500
max,29.914600


Seven-day average temperature: min=29.91, mean=29.91, max=29.91 °C


In [11]:
# Request 2/4: total hours above the selected crop threshold.
exceedance_response = run_heatmap(
    'exceedance',
    analytic_type='exceedance',
    threshold=HEAT_THRESHOLD_C,
    direction='above',
)
print(json.dumps(exceedance_response['result'].get('stats_data', {}), indent=2))

Submitted -> activity_id=b8f25479-e5e6-46b9-8744-26ddb8a8b7e9
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Activity ID: b8f25479-e5e6-46b9-8744-26ddb8a8b7e9
Result keys: ['map_data', 'stats_data']
Returned tiles: 68
{
  "activity_id": "b8f25479-e5e6-46b9-8744-26ddb8a8b7e9",
  "analytic_type": "exceedance",
  "units": "hour",
  "n_cells": 68,
  "min": 6.0,
  "max": 6.0,
  "mean": 6.0
}


In [12]:
# Request 3/4: longest continuous run above the same threshold.
persistence_response = run_heatmap(
    'persistence',
    analytic_type='persistence',
    threshold=HEAT_THRESHOLD_C,
    direction='above',
)
print(json.dumps(persistence_response['result'].get('stats_data', {}), indent=2))

Submitted -> activity_id=9bb4bef3-e8b4-441e-9551-446d6190ecb4
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Activity ID: 9bb4bef3-e8b4-441e-9551-446d6190ecb4
Result keys: ['map_data', 'stats_data']
Returned tiles: 68
{
  "activity_id": "9bb4bef3-e8b4-441e-9551-446d6190ecb4",
  "analytic_type": "persistence",
  "units": "hour",
  "n_cells": 68,
  "min": 2.0,
  "max": 2.0,
  "mean": 2.0
}


In [13]:
# Request 4/4: UTC hour when each tile reaches peak heat.
peak_time_response = run_heatmap('time_of_measure', analytic_type='time_of_measure')
print(json.dumps(peak_time_response['result'].get('stats_data', {}), indent=2))

Submitted -> activity_id=5f4f76a7-9934-455e-9005-c169b4954178
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Activity ID: 5f4f76a7-9934-455e-9005-c169b4954178
Result keys: ['map_data', 'stats_data']
Returned tiles: 68
{
  "activity_id": "5f4f76a7-9934-455e-9005-c169b4954178",
  "analytic_type": "time_of_measure",
  "units": "hour",
  "n_cells": 68,
  "min": 15.0,
  "max": 15.0,
  "mean": 15.0
}


In [14]:
expected_analyses = {'tcm', 'exceedance', 'persistence', 'time_of_measure'}
missing_responses = expected_analyses - set(responses)
if missing_responses:
    raise ValueError(f'Run the missing request cells first: {sorted(missing_responses)}')

validation_rows = []
tile_frames = {}
for name, response in responses.items():
    result = response.get('result', {})
    features = result.get('map_data', {}).get('features', [])
    frame = tile_dataframe(response, name)
    tile_frames[name] = frame
    validation_rows.append({
        'analysis': name,
        'activity_id_present': bool(response.get('activity_id')),
        'tile_count': len(features),
        'value_field': frame['value_key'].iloc[0] if len(frame) else None,
        'numeric_count': int(frame['value'].notna().sum()),
        'null_count': int(frame['value'].isna().sum()),
        'min': frame['value'].min(),
        'mean': frame['value'].mean(),
        'max': frame['value'].max(),
        'reported_units': result.get('stats_data', {}).get('units'),
    })

response_validation_df = pd.DataFrame(validation_rows).set_index('analysis')
if (response_validation_df['tile_count'] == 0).any():
    raise ValueError('At least one heatmap returned no tiles.')
if (response_validation_df['null_count'] > 0).any():
    print('Warning: at least one response contains null tile values.')
display(response_validation_df)

,activity_id_present,tile_count,value_field,numeric_count,null_count,min,mean,max,reported_units
analysis,,,,,,,,,
tcm,True,68,average_temperature,68,0,29.9143,29.914446,29.9146,NaN
exceedance,True,68,value,68,0,6.0000,6.000000,6.0000,hour
persistence,True,68,value,68,0,2.0000,2.000000,2.0000,hour
time_of_measure,True,68,value,68,0,15.0000,15.000000,15.0000,hour


In [15]:
render_heatmap(
    exceedance_response,
    f'Hours above {HEAT_THRESHOLD_C:.1f} °C',
    'hours',
    colors=('white', 'orange', 'darkred'),
)
render_heatmap(
    persistence_response,
    f'Longest continuous run above {HEAT_THRESHOLD_C:.1f} °C',
    'hours',
    colors=('white', 'gold', 'purple'),
)
render_heatmap(
    peak_time_response,
    'UTC hour of peak heat',
    'UTC hour',
    colors=('navy', 'cyan', 'yellow', 'red'),
)

Hours above 37.8 °C: min=6.00, mean=6.00, max=6.00 hours


Longest continuous run above 37.8 °C: min=2.00, mean=2.00, max=2.00 hours


UTC hour of peak heat: min=15.00, mean=15.00, max=15.00 UTC hour


In [16]:
# Weight boundary tiles by their relative overlap with the farm AOI.
# For this very small AOI, degree-based relative areas are sufficient for weighting.
aoi_geometry = shape(FARM_AOI['features'][0]['geometry'])

def area_weighted_mean(response):
    result = response['result']
    key = feature_value_key(result)
    weighted_sum = 0.0
    total_weight = 0.0
    for feature in result['map_data']['features']:
        raw_value = feature.get('properties', {}).get(key)
        if raw_value is None:
            continue
        overlap = shape(feature['geometry']).intersection(aoi_geometry).area
        if overlap <= 0:
            continue
        weighted_sum += float(raw_value) * overlap
        total_weight += overlap
    return weighted_sum / total_weight if total_weight else None

peak_utc_hour = area_weighted_mean(peak_time_response)
peak_local_hour = None
if peak_utc_hour is not None:
    reference_utc = datetime.combine(
        END_DATE,
        datetime.min.time(),
        tzinfo=timezone.utc,
    ) + timedelta(hours=round(peak_utc_hour))
    peak_local_hour = reference_utc.astimezone(ZoneInfo(FARM_TIMEZONE)).hour

farm_heat_summary = pd.Series({
    'provider': 'FortyGuard',
    'farm_name': FARM_NAME,
    'window_start': START_DATE.isoformat(),
    'window_end': END_DATE.isoformat(),
    'selected_crop_id': SELECTED_CROP_ID,
    'heat_threshold_c': HEAT_THRESHOLD_C,
    'area_weighted_temperature_c': area_weighted_mean(tcm_response),
    'area_weighted_exceedance_hours': area_weighted_mean(exceedance_response),
    'area_weighted_persistence_hours': area_weighted_mean(persistence_response),
    'area_weighted_peak_hour_utc': peak_utc_hour,
    'approx_peak_hour_local': peak_local_hour,
    'timezone': FARM_TIMEZONE,
    'granularity_m': GRANULARITY_M,
})
display(farm_heat_summary.to_frame('value'))

,value
provider,FortyGuard
farm_name,Plainview demonstration farm
window_start,2026-08-19
window_end,2026-08-25
selected_crop_id,upland_cotton
heat_threshold_c,37.8
area_weighted_temperature_c,29.914448
area_weighted_exceedance_hours,6.0
area_weighted_persistence_hours,2.0
area_weighted_peak_hour_utc,15.0


In [17]:
# Save a sanitized local cache. The API key is never part of these response objects.
cache_dir = PROJECT_ROOT / 'data' / 'fortyguard-cache'
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = cache_dir / f'plainview_heatmaps_{START_DATE}_{END_DATE}.json'

cache_payload = {
    'provider': 'FortyGuard',
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
    'live_or_cached': 'live',
    'farm': {
        'name': FARM_NAME,
        'latitude': FARM_LAT,
        'longitude': FARM_LON,
        'timezone': FARM_TIMEZONE,
        'polygon_aoi': FARM_AOI,
    },
    'request_context': {
        'start_date': START_DATE.isoformat(),
        'end_date': END_DATE.isoformat(),
        'filter_type': 4,
        'granularity_m': GRANULARITY_M,
        'selected_crop_id': SELECTED_CROP_ID,
        'threshold_c': HEAT_THRESHOLD_C,
        'direction': 'above',
    },
    'normalized_summary': farm_heat_summary.to_dict(),
    'responses': responses,
}
cache_path.write_text(json.dumps(cache_payload, indent=2, default=str), encoding='utf-8')
print('Saved sanitized cache:', cache_path)
print('File size:', round(cache_path.stat().st_size / 1024, 1), 'KB')

Saved sanitized cache: C:\Users\sumai\OneDrive\Documents\ChatGPT\CropSage\data\fortyguard-cache\plainview_heatmaps_2026-08-19_2026-08-25.json
File size: 293.9 KB


## CropSage integration decision

If the four live requests succeed and the returned values are plausible, CropSage will use FortyGuard as its central local heat source.

Values intended for the recommendation engine:

- Area-weighted recent temperature context
- Crop-specific hours above the catalog heat threshold
- Longest continuous heat exposure above that threshold
- Approximate local hour of peak heat
- Spatial variation across the farm tiles
- Provider, time window, granularity, activity IDs, and live/cached status

Important limits:

- Heat thresholds vary by crop, variety, growth stage, soil moisture, and management.
- Exceedance is a count of hours, not degree-hours.
- Persistence is the longest continuous run, not total exposure.
- A 100 m provider tile is not an on-farm sensor measurement.
- The short forecast is an operational warning and must not be presented as a seasonal crop forecast.
- The recommendation engine must keep suitability, risk, and confidence separate.

# All-Crop Multi-Window Heat Tests

The completed cotton run above proved the live endpoint shapes. The following sections expand the test to all 22 catalog records over four comparable windows ending on the same date:

- 7 days (1 week)
- 14 days (2 weeks)
- 21 days (3 weeks)
- 30 days (one API-safe month)

Important interpretation rules:

- All 22 records appear in every result table. Long-grain rice is retained but marked regionally ineligible for the Texas Plains.
- TCM temperature evidence is shared by all crops.
- Only 10 Plains crops have a numeric heat threshold.
- Those 10 crops use seven unique thresholds, so crops sharing a threshold reuse one API result.
- Crops without numeric thresholds remain evaluated for regional and optimal-temperature fit; no threshold is invented.
- Persistence remains observational until CropSage documents a defensible duration-to-penalty policy.
- Raw peak hour is retained, but its time-zone interpretation remains unverified and is not scored.

Each completed response is cached immediately. Rerunning a section loads the cache instead of consuming credits again.

In [ ]:
TEST_REGION_ID = 'plains'
ALL_CROP_END_DATE = END_DATE
ALL_CROP_WINDOWS = {
    'one_week': 7,
    'two_weeks': 14,
    'three_weeks': 21,
    'one_month_30_days': 30,
}

all_crop_rows = []
for crop in catalog['crops']:
    optimum = crop.get('optimal_temperature_range') or {}
    threshold = crop.get('heat_stress_threshold') or {}
    all_crop_rows.append({
        'crop_id': crop['crop_id'],
        'crop': crop['common_name'],
        'regionally_eligible': TEST_REGION_ID in crop.get('supported_texas_regions', []),
        'optimal_min_c': optimum.get('min_c'),
        'optimal_max_c': optimum.get('max_c'),
        'heat_threshold_c': threshold.get('value_c'),
        'threshold_scoring_use': threshold.get('scoring_use'),
        'threshold_evidence_status': threshold.get('evidence_status'),
        'catalog_confidence': crop.get('confidence'),
        'record_status': crop.get('record_status'),
    })

all_crops_test_df = pd.DataFrame(all_crop_rows)
numeric_thresholds = sorted(
    float(value)
    for value in all_crops_test_df.loc[
        all_crops_test_df['regionally_eligible'],
        'heat_threshold_c',
    ].dropna().unique()
)

threshold_groups_df = (
    all_crops_test_df[
        all_crops_test_df['regionally_eligible']
        & all_crops_test_df['heat_threshold_c'].notna()
    ]
    .groupby('heat_threshold_c', as_index=False)
    .agg(
        crop_count=('crop_id', 'count'),
        crops=('crop', lambda values: ', '.join(values)),
    )
    .sort_values('heat_threshold_c')
)

print('Catalog records included:', len(all_crops_test_df))
print('Texas Plains eligible:', int(all_crops_test_df['regionally_eligible'].sum()))
print('Numeric-threshold crops:', int(all_crops_test_df['heat_threshold_c'].notna().sum()))
print('Unique threshold requests:', numeric_thresholds)
print('Requests per complete window:', 2 + 2 * len(numeric_thresholds))
display(threshold_groups_df)
display(all_crops_test_df)

In [ ]:
MULTI_WINDOW_CACHE_DIR = PROJECT_ROOT / 'data' / 'fortyguard-cache' / 'multi-window'
MULTI_WINDOW_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_MULTI_WINDOW_REQUESTS = True

def threshold_token(threshold):
    return 'none' if threshold is None else f'{float(threshold):.1f}'.replace('.', 'p')

def request_cache_path(start_date, end_date, analytic_type, threshold=None):
    window_dir = MULTI_WINDOW_CACHE_DIR / f'{start_date}_{end_date}'
    window_dir.mkdir(parents=True, exist_ok=True)
    return window_dir / f'{analytic_type}_threshold_{threshold_token(threshold)}.json'

def load_baseline_response_if_available(start_date, end_date, analytic_type, threshold=None):
    baseline_path = PROJECT_ROOT / 'data' / 'fortyguard-cache' / f'plainview_heatmaps_{start_date}_{end_date}.json'
    if not baseline_path.exists():
        return None
    payload = json.loads(baseline_path.read_text(encoding='utf-8'))
    baseline_threshold = float(payload.get('request_context', {}).get('threshold_c'))
    if analytic_type in ('tcm', 'time_of_measure'):
        return payload.get('responses', {}).get(analytic_type)
    if threshold is not None and abs(float(threshold) - baseline_threshold) < 1e-9:
        return payload.get('responses', {}).get(analytic_type)
    return None

def get_or_create_window_response(start_date, end_date, analytic_type, threshold=None):
    cache_path = request_cache_path(start_date, end_date, analytic_type, threshold)
    if cache_path.exists():
        saved = json.loads(cache_path.read_text(encoding='utf-8'))
        print(f'Cache hit: {cache_path.name}')
        return saved['response'], 'cache'

    baseline_response = load_baseline_response_if_available(
        start_date, end_date, analytic_type, threshold
    )
    if baseline_response is not None:
        print(f'Reusing completed baseline: {analytic_type}, threshold={threshold}')
        return baseline_response, 'baseline_cache'

    if not ALLOW_LIVE_MULTI_WINDOW_REQUESTS:
        raise RuntimeError(
            f'No cache for {analytic_type}, threshold={threshold}. '
            'Set ALLOW_LIVE_MULTI_WINDOW_REQUESTS=True to submit it.'
        )

    kwargs = {
        'polygon_aoi': FARM_AOI,
        'start_date': start_date.isoformat(),
        'end_date': end_date.isoformat(),
        'filter_type': 4,
        'granularity': GRANULARITY_M,
        'analytic_type': analytic_type,
        'poll_interval': 3.0,
        'timeout': 600.0,
        'verbose': True,
    }
    if threshold is not None:
        kwargs['threshold'] = float(threshold)
        kwargs['direction'] = 'above'

    print(f'Live request: {analytic_type}, threshold={threshold}, {start_date} to {end_date}')
    response = client.create_heatmap(**kwargs)
    saved = {
        'provider': 'FortyGuard',
        'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
        'request': {
            'start_date': start_date.isoformat(),
            'end_date': end_date.isoformat(),
            'analytic_type': analytic_type,
            'threshold_c': threshold,
            'direction': 'above' if threshold is not None else None,
            'granularity_m': GRANULARITY_M,
        },
        'response': response,
    }
    cache_path.write_text(json.dumps(saved, indent=2, default=str), encoding='utf-8')
    print('Cached:', cache_path)
    return response, 'live'

def response_value_key(response):
    features = response['result'].get('map_data', {}).get('features', [])
    if not features:
        raise ValueError('No map features returned.')
    keys = features[0].get('properties', {}).keys()
    for candidate in ('average_temperature', 'temperature', 'value'):
        if candidate in keys:
            return candidate
    raise KeyError(f'No supported tile value. Available fields: {list(keys)}')

def response_area_weighted_mean(response):
    key = response_value_key(response)
    aoi_geometry = shape(FARM_AOI['features'][0]['geometry'])
    weighted_sum = 0.0
    total_weight = 0.0
    for feature in response['result']['map_data']['features']:
        raw_value = feature.get('properties', {}).get(key)
        if raw_value is None:
            continue
        overlap = shape(feature['geometry']).intersection(aoi_geometry).area
        if overlap > 0:
            weighted_sum += float(raw_value) * overlap
            total_weight += overlap
    return weighted_sum / total_weight if total_weight else None

def run_all_crop_window(window_name, days):
    end_date = ALL_CROP_END_DATE
    start_date = end_date - timedelta(days=days - 1)
    print(f'\n=== {window_name}: {start_date} to {end_date} ({days} days) ===')
    bundle = {
        'window_name': window_name,
        'days': days,
        'start_date': start_date,
        'end_date': end_date,
        'responses': {},
        'sources': {},
        'thresholds': {},
    }

    for analytic_type in ('tcm', 'time_of_measure'):
        response, source = get_or_create_window_response(
            start_date, end_date, analytic_type
        )
        bundle['responses'][analytic_type] = response
        bundle['sources'][analytic_type] = source

    for threshold in numeric_thresholds:
        threshold_results = {}
        for analytic_type in ('exceedance', 'persistence'):
            response, source = get_or_create_window_response(
                start_date, end_date, analytic_type, threshold
            )
            threshold_results[analytic_type] = response
            threshold_results[f'{analytic_type}_source'] = source
        bundle['thresholds'][float(threshold)] = threshold_results

    print(f'Completed window: {window_name}')
    return bundle

print('Multi-window helpers ready.')

In [ ]:
def summarize_all_crop_window(bundle):
    tcm_response = bundle['responses']['tcm']
    peak_response = bundle['responses']['time_of_measure']
    tcm_features = tcm_response['result']['map_data']['features']
    tcm_properties = [feature.get('properties', {}) for feature in tcm_features]

    period_mean_c = response_area_weighted_mean(tcm_response)
    period_min_c = min(
        float(properties['min_temperature'])
        for properties in tcm_properties
        if properties.get('min_temperature') is not None
    )
    period_max_c = max(
        float(properties['max_temperature'])
        for properties in tcm_properties
        if properties.get('max_temperature') is not None
    )
    peak_hour_raw = response_area_weighted_mean(peak_response)

    validation_rows = []
    response_items = [
        ('tcm', None, tcm_response, bundle['sources']['tcm']),
        ('time_of_measure', None, peak_response, bundle['sources']['time_of_measure']),
    ]
    for threshold, results in bundle['thresholds'].items():
        response_items.extend([
            ('exceedance', threshold, results['exceedance'], results['exceedance_source']),
            ('persistence', threshold, results['persistence'], results['persistence_source']),
        ])

    for analytic_type, threshold, response, source in response_items:
        key = response_value_key(response)
        values = [
            pd.to_numeric(feature.get('properties', {}).get(key), errors='coerce')
            for feature in response['result']['map_data']['features']
        ]
        series = pd.Series(values, dtype='float64')
        validation_rows.append({
            'analytic_type': analytic_type,
            'threshold_c': threshold,
            'activity_id': response.get('activity_id'),
            'source': source,
            'tile_count': len(series),
            'numeric_count': int(series.notna().sum()),
            'null_count': int(series.isna().sum()),
            'min': series.min(),
            'mean': series.mean(),
            'max': series.max(),
            'unique_values': int(series.nunique(dropna=True)),
            'reported_units': response['result'].get('stats_data', {}).get('units'),
        })
    validation_df = pd.DataFrame(validation_rows)

    threshold_rows = []
    for threshold, results in sorted(bundle['thresholds'].items()):
        matching_crops = all_crops_test_df.loc[
            all_crops_test_df['heat_threshold_c'].eq(threshold)
            & all_crops_test_df['regionally_eligible'],
            'crop',
        ].tolist()
        threshold_rows.append({
            'threshold_c': threshold,
            'crops': ', '.join(matching_crops),
            'exceedance_hours': response_area_weighted_mean(results['exceedance']),
            'persistence_hours': response_area_weighted_mean(results['persistence']),
            'exceedance_activity_id': results['exceedance'].get('activity_id'),
            'persistence_activity_id': results['persistence'].get('activity_id'),
        })
    threshold_results_df = pd.DataFrame(threshold_rows)

    crop_rows = []
    threshold_lookup = threshold_results_df.set_index('threshold_c').to_dict('index')
    for crop in all_crop_rows:
        eligible = bool(crop['regionally_eligible'])
        optimal_min = crop['optimal_min_c']
        optimal_max = crop['optimal_max_c']
        if not eligible:
            temperature_fit = 'regionally_ineligible'
            distance_from_optimum_c = None
        elif period_mean_c < optimal_min:
            temperature_fit = 'below_optimal_mean'
            distance_from_optimum_c = period_mean_c - optimal_min
        elif period_mean_c > optimal_max:
            temperature_fit = 'above_optimal_mean'
            distance_from_optimum_c = period_mean_c - optimal_max
        else:
            temperature_fit = 'within_optimal_mean'
            distance_from_optimum_c = 0.0

        threshold = crop['heat_threshold_c']
        threshold_result = threshold_lookup.get(float(threshold)) if pd.notna(threshold) else None
        exceedance_hours = threshold_result['exceedance_hours'] if threshold_result else None
        persistence_hours = threshold_result['persistence_hours'] if threshold_result else None

        if not eligible:
            heat_action = 'excluded_for_plains'
        elif threshold_result is None:
            heat_action = 'no_numeric_threshold_do_not_invent'
        elif exceedance_hours == 0:
            heat_action = 'no_threshold_exceedance'
        elif crop['threshold_scoring_use'] == 'soft_penalty':
            heat_action = 'soft_penalty_candidate'
        else:
            heat_action = 'informational_warning_only'

        crop_rows.append({
            **crop,
            'period_mean_c': period_mean_c,
            'period_min_c': period_min_c,
            'period_max_c': period_max_c,
            'temperature_fit': temperature_fit,
            'distance_from_optimum_c': distance_from_optimum_c,
            'exceedance_hours': exceedance_hours,
            'persistence_hours': persistence_hours,
            'exceedance_fraction_of_window': (
                exceedance_hours / (bundle['days'] * 24)
                if exceedance_hours is not None else None
            ),
            'heat_action': heat_action,
        })
    crop_results_df = pd.DataFrame(crop_rows)

    period_summary = pd.Series({
        'window_name': bundle['window_name'],
        'window_days': bundle['days'],
        'window_start': bundle['start_date'].isoformat(),
        'window_end': bundle['end_date'].isoformat(),
        'catalog_crop_count': len(crop_results_df),
        'plains_eligible_count': int(crop_results_df['regionally_eligible'].sum()),
        'numeric_threshold_crop_count': int(crop_results_df['heat_threshold_c'].notna().sum()),
        'unique_threshold_count': len(threshold_results_df),
        'request_count': len(validation_df),
        'tile_count': len(tcm_features),
        'null_values_across_requests': int(validation_df['null_count'].sum()),
        'area_weighted_mean_temperature_c': period_mean_c,
        'minimum_temperature_c': period_min_c,
        'maximum_temperature_c': period_max_c,
        'peak_hour_raw': peak_hour_raw,
        'peak_hour_timezone_status': 'unverified',
        'granularity_m': GRANULARITY_M,
    })
    return period_summary, validation_df, threshold_results_df, crop_results_df

def display_all_crop_window_report(bundle):
    period_summary, validation_df, threshold_df, crop_df = summarize_all_crop_window(bundle)
    print(f"\n{bundle['window_name'].replace('_', ' ').title()} summary")
    display(period_summary.to_frame('value'))
    print('\nRequest validation')
    display(validation_df)
    print('\nUnique threshold results')
    display(threshold_df)
    print('\nAll 22 crop results')
    display(crop_df[[
        'crop_id', 'crop', 'regionally_eligible',
        'optimal_min_c', 'optimal_max_c', 'period_mean_c',
        'temperature_fit', 'heat_threshold_c', 'threshold_scoring_use',
        'exceedance_hours', 'persistence_hours',
        'exceedance_fraction_of_window', 'heat_action',
        'threshold_evidence_status', 'catalog_confidence',
    ]])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(threshold_df['threshold_c'], threshold_df['exceedance_hours'], marker='o')
    axes[0].set_title(f"{bundle['days']}-day threshold exceedance")
    axes[0].set_xlabel('Crop threshold (°C)')
    axes[0].set_ylabel('Area-weighted exceedance (hours)')
    axes[0].grid(alpha=0.3)
    axes[1].plot(threshold_df['threshold_c'], threshold_df['persistence_hours'], marker='o', color='purple')
    axes[1].set_title(f"{bundle['days']}-day threshold persistence")
    axes[1].set_xlabel('Crop threshold (°C)')
    axes[1].set_ylabel('Longest continuous run (hours)')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    summary_dir = MULTI_WINDOW_CACHE_DIR / f"{bundle['start_date']}_{bundle['end_date']}"
    summary_path = summary_dir / 'all_crop_summary.json'
    summary_payload = {
        'period_summary': period_summary.to_dict(),
        'request_validation': validation_df.to_dict(orient='records'),
        'threshold_results': threshold_df.to_dict(orient='records'),
        'crop_results': crop_df.to_dict(orient='records'),
    }
    summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
    print('Saved detailed summary:', summary_path)
    return period_summary, validation_df, threshold_df, crop_df

def render_window_threshold_maps(bundle, threshold_c=37.8):
    threshold_c = float(threshold_c)
    if threshold_c not in bundle['thresholds']:
        raise ValueError(f'Choose one of: {sorted(bundle["thresholds"])}')
    render_heatmap(
        bundle['responses']['tcm'],
        f"{bundle['days']}-day average temperature",
        '°C',
    )
    render_heatmap(
        bundle['thresholds'][threshold_c]['exceedance'],
        f"{bundle['days']}-day hours above {threshold_c:.1f} °C",
        'hours',
        colors=('white', 'orange', 'darkred'),
    )
    render_heatmap(
        bundle['thresholds'][threshold_c]['persistence'],
        f"{bundle['days']}-day persistence above {threshold_c:.1f} °C",
        'hours',
        colors=('white', 'gold', 'purple'),
    )

print('Multi-window reporting helpers ready.')

## Test A — All 22 Crops Over 1 Week

Window length: 7 days. The completed TCM, peak-hour, and cotton 37.8 C responses are reused from the existing baseline cache. The remaining unique-threshold responses are requested and cached.

In [ ]:
one_week_bundle = run_all_crop_window('one_week', ALL_CROP_WINDOWS['one_week'])

In [ ]:
(
    one_week_summary,
    one_week_validation_df,
    one_week_threshold_df,
    one_week_crop_df,
) = display_all_crop_window_report(one_week_bundle)

In [ ]:
# Optional detailed maps. Change to any threshold listed in threshold_groups_df.
render_window_threshold_maps(one_week_bundle, threshold_c=37.8)

## Test B — All 22 Crops Over 2 Weeks

Window length: 14 days ending on the same date as Test A. This section performs the complete 16-request threshold-group suite and creates a detailed 22-crop comparison.

In [ ]:
two_week_bundle = run_all_crop_window('two_weeks', ALL_CROP_WINDOWS['two_weeks'])

In [ ]:
(
    two_week_summary,
    two_week_validation_df,
    two_week_threshold_df,
    two_week_crop_df,
) = display_all_crop_window_report(two_week_bundle)

In [ ]:
# Optional detailed maps.
render_window_threshold_maps(two_week_bundle, threshold_c=37.8)

## Test C — All 22 Crops Over 3 Weeks

Window length: 21 days ending on the same date. Results remain normalized by both raw hours and fraction of the observation window so periods can be compared fairly.

In [ ]:
three_week_bundle = run_all_crop_window('three_weeks', ALL_CROP_WINDOWS['three_weeks'])

In [ ]:
(
    three_week_summary,
    three_week_validation_df,
    three_week_threshold_df,
    three_week_crop_df,
) = display_all_crop_window_report(three_week_bundle)

In [ ]:
# Optional detailed maps.
render_window_threshold_maps(three_week_bundle, threshold_c=37.8)

## Test D — All 22 Crops Over One Month

Window length: 30 consecutive days ending on the same date. Thirty days is used instead of a variable calendar month to remain safely inside the documented approximately 31-day API range.

In [ ]:
one_month_bundle = run_all_crop_window(
    'one_month_30_days',
    ALL_CROP_WINDOWS['one_month_30_days'],
)

In [ ]:
(
    one_month_summary,
    one_month_validation_df,
    one_month_threshold_df,
    one_month_crop_df,
) = display_all_crop_window_report(one_month_bundle)

In [ ]:
# Optional detailed maps.
render_window_threshold_maps(one_month_bundle, threshold_c=37.8)

## Cross-Window Comparison — Run After All Four Tests

This final section compares temperature context and threshold exposure across 7, 14, 21, and 30 days. Raw hours naturally grow with a longer window, so the comparison also uses exceedance fraction of window hours.

In [ ]:
window_summaries = pd.DataFrame([
    one_week_summary,
    two_week_summary,
    three_week_summary,
    one_month_summary,
]).set_index('window_name')
display(window_summaries)

cross_window_crops_df = pd.concat([
    one_week_crop_df.assign(window_days=7),
    two_week_crop_df.assign(window_days=14),
    three_week_crop_df.assign(window_days=21),
    one_month_crop_df.assign(window_days=30),
], ignore_index=True)

display(cross_window_crops_df[[
    'window_days', 'crop_id', 'crop', 'regionally_eligible',
    'period_mean_c', 'temperature_fit', 'heat_threshold_c',
    'exceedance_hours', 'persistence_hours',
    'exceedance_fraction_of_window', 'heat_action',
]])

comparison_path = MULTI_WINDOW_CACHE_DIR / 'cross_window_all_crop_comparison.csv'
cross_window_crops_df.to_csv(comparison_path, index=False)
print('Saved cross-window comparison:', comparison_path)